In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",          17.21, 12.615),
    ("元Blend(LGB/PLS/Ridge)",    14.10, 12.647),
    ("Mixup seed42",              18.04, 11.800),
    ("SafeMulti3(ensemble)",      17.64, 11.870),
    ("G2_alpha015",               18.43, 11.935),
    ("seed35 LGB",                16.79, 11.644),
    ("C_plsonly2 seed42",         19.60, 11.788),
    ("LGB35×0.75+PLS2_s0×0.25",  16.67, 11.428),
]

print("=" * 60)
print("🏆 ブレンド精密最適化")
print("=" * 60)
print(f"""
  ★ LB=11.428 (LGB35×0.75 + PLS2_s0×0.25)
  
  今回の目標:
  1. w=0.70~0.80を0.01刻みで精密探索
  2. 異なるPLS seed/成分数でのブレンド
  3. 3モデルブレンド (LGB35 + LGB_X + PLS)
  4. 異なるLGB seedでのブレンド
""")

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values
y_true = np.expm1(y_train_log)


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 2. LGBパイプライン
# ============================================================
def run_lgb(seed, verbose=False):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_aug = apply_snv(X_aug)
        d1_aug  = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va  = apply_snv(X_va)
        d1_va   = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te  = apply_snv(X_test_raw)
        d1_te   = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va  = pca.transform(snv_va)
        pc_te  = pca.transform(snv_te)

        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[it], axis=1).reshape(-1, 1)

        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  LGB seed={seed:4d}  OOF={oof_rmse:.4f}")
    return {'seed': seed, 'oof': oof_rmse, 'test_pred': final_pred,
            'oof_pred': oof_pred, 'type': 'LGB'}


# ============================================================
# 3. PLS-onlyパイプライン
# ============================================================
def run_pls(n_comp=2, seed=42, verbose=False):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_orig = apply_snv(X_tr)
        snv_aug  = apply_snv(X_aug)
        snv_va   = apply_snv(X_va)
        snv_te   = apply_snv(X_test_raw)

        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(snv_orig, y_tr)

        ps_aug = pls.transform(snv_aug)
        ps_va  = pls.transform(snv_va)
        ps_te  = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va  = pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te  = pls.predict(snv_te).ravel().reshape(-1,1)

        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(ps_orig)

        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(ps_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(ps_te, 5)
        knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        ft = np.hstack([ps_aug, pp_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([ps_va, pp_va, knn_va, r_va, s_va])
        fe = np.hstack([ps_te, pp_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=4, num_leaves=15,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  PLS{n_comp} seed={seed:4d}  OOF={oof_rmse:.4f}")
    return {'name': f'PLS{n_comp}_s{seed}', 'oof': oof_rmse,
            'test_pred': final_pred, 'oof_pred': oof_pred, 'type': 'PLS'}


# ============================================================
# 4. モデル構築（キャッシュ）
# ============================================================
print(f"\n{'='*60}")
print("🔧 ベースモデル構築")
print(f"{'='*60}")

# LGBモデル群
lgb_seeds = [35, 39, 33, 42, 25, 29, 34, 28]
lgb_models = {}
for s in lgb_seeds:
    r = run_lgb(s, verbose=True)
    lgb_models[s] = r

# PLSモデル群（成分数とseed両方変える）
pls_configs = [
    (2, 0), (2, 42), (2, 33), (2, 35), (2, 38),
    (1, 0), (1, 42),  # 1成分
    (3, 0), (3, 42),  # 3成分
]
pls_models = {}
for nc, s in pls_configs:
    r = run_pls(n_comp=nc, seed=s, verbose=True)
    pls_models[(nc, s)] = r


# ============================================================
# 5. 2モデルブレンド精密探索
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験1: 2モデルブレンド精密探索")
print(f"{'='*60}")

blend2_results = []

for lgb_seed, lgb_r in lgb_models.items():
    for (nc, ps), pls_r in pls_models.items():
        for w in np.arange(0.60, 0.91, 0.01):
            oof_bl = w * lgb_r['oof_pred'] + (1-w) * pls_r['oof_pred']
            rmse = np.sqrt(mean_squared_error(y_true, oof_bl))
            pred = w * lgb_r['test_pred'] + (1-w) * pls_r['test_pred']
            blend2_results.append({
                'name': f"LGB{lgb_seed}×{w:.2f}+PLS{nc}_s{ps}×{1-w:.2f}",
                'lgb_seed': lgb_seed, 'pls_nc': nc, 'pls_seed': ps,
                'w': w, 'oof': rmse,
                'test_pred': pred, 'oof_pred': oof_bl,
            })

blend2_results.sort(key=lambda x: x['oof'])

print(f"  全{len(blend2_results)}組み合わせを探索")
print(f"\n  Top 20:")
print(f"  {'Name':<45s} {'OOF':>8s}")
print(f"  {'─'*45} {'─'*8}")
for b in blend2_results[:20]:
    print(f"  {b['name']:<45s} {b['oof']:>8.4f}")

# 勝ちパターンの分析
print(f"\n  --- 勝ちパターン分析 ---")
top50 = blend2_results[:50]

# LGB seedの分布
from collections import Counter
lgb_counter = Counter(b['lgb_seed'] for b in top50)
print(f"  Top50のLGB seed分布: {dict(lgb_counter.most_common())}")

# PLS configの分布
pls_counter = Counter((b['pls_nc'], b['pls_seed']) for b in top50)
print(f"  Top50のPLS config分布: {dict(pls_counter.most_common())}")

# w値の分布
w_values = [b['w'] for b in top50]
print(f"  Top50のw値: mean={np.mean(w_values):.3f}, "
      f"min={min(w_values):.2f}, max={max(w_values):.2f}")


# ============================================================
# 6. 3モデルブレンド探索
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験2: 3モデルブレンド探索")
print("   LGB_A + LGB_B + PLS")
print(f"{'='*60}")

blend3_results = []

# LGBのTop4 × PLSのTop3
top_lgb_seeds = [35, 39, 33, 42]
top_pls_configs = [(2, 0), (2, 42), (1, 0)]

from itertools import combinations

for (s1, s2) in combinations(top_lgb_seeds, 2):
    for (nc, ps) in top_pls_configs:
        lgb1 = lgb_models[s1]
        lgb2 = lgb_models[s2]
        pls_r = pls_models[(nc, ps)]

        # w1 + w2 + w3 = 1, w3 = PLS weight
        for w3 in np.arange(0.10, 0.36, 0.05):
            remaining = 1.0 - w3
            for ratio in np.arange(0.3, 0.8, 0.1):
                w1 = remaining * ratio
                w2 = remaining * (1 - ratio)
                oof_bl = (w1 * lgb1['oof_pred'] +
                          w2 * lgb2['oof_pred'] +
                          w3 * pls_r['oof_pred'])
                rmse = np.sqrt(mean_squared_error(y_true, oof_bl))
                pred = (w1 * lgb1['test_pred'] +
                        w2 * lgb2['test_pred'] +
                        w3 * pls_r['test_pred'])
                blend3_results.append({
                    'name': (f"LGB{s1}×{w1:.2f}+LGB{s2}×{w2:.2f}"
                             f"+PLS{nc}_s{ps}×{w3:.2f}"),
                    'seeds': (s1, s2), 'pls': (nc, ps),
                    'weights': (w1, w2, w3),
                    'oof': rmse, 'test_pred': pred, 'oof_pred': oof_bl,
                })

blend3_results.sort(key=lambda x: x['oof'])

print(f"  全{len(blend3_results)}組み合わせを探索")
print(f"\n  Top 15:")
print(f"  {'Name':<55s} {'OOF':>8s}")
print(f"  {'─'*55} {'─'*8}")
for b in blend3_results[:15]:
    print(f"  {b['name']:<55s} {b['oof']:>8.4f}")

# 2モデル vs 3モデル比較
best2 = blend2_results[0]
best3 = blend3_results[0]
print(f"\n  ベスト2モデル: OOF={best2['oof']:.4f}")
print(f"  ベスト3モデル: OOF={best3['oof']:.4f}")
print(f"  改善幅: {best2['oof'] - best3['oof']:+.4f}")


# ============================================================
# 7. LB=11.428の再現ブレンドとの予測差分分析
# ============================================================
print(f"\n{'='*60}")
print("📊 11.428ブレンドとの差分分析")
print(f"{'='*60}")

# 11.428 = LGB35×0.75 + PLS2_s0×0.25
ref_pred = (0.75 * lgb_models[35]['test_pred'] +
            0.25 * pls_models[(2, 0)]['test_pred'])

print(f"  {'候補':<45s} {'vs 11.428 RMSD':>15s} {'corr':>7s}")
print(f"  {'─'*45} {'─'*15} {'─'*7}")

# Top5 の2モデルブレンド
for b in blend2_results[:5]:
    diff = b['test_pred'] - ref_pred
    rmsd = np.sqrt(np.mean(diff**2))
    corr = np.corrcoef(b['test_pred'], ref_pred)[0, 1]
    print(f"  {b['name']:<45s} {rmsd:>15.4f} {corr:>7.4f}")

# Top5 の3モデルブレンド
for b in blend3_results[:5]:
    diff = b['test_pred'] - ref_pred
    rmsd = np.sqrt(np.mean(diff**2))
    corr = np.corrcoef(b['test_pred'], ref_pred)[0, 1]
    print(f"  {b['name']:<45s} {rmsd:>15.4f} {corr:>7.4f}")


# ============================================================
# 8. 提出ファイル作成
# ============================================================
print(f"\n{'='*60}")
print("📁 提出ファイル作成")
print(f"{'='*60}")

all_subs = {}

# 11.428再現
ref_out = submit_template.copy()
ref_out[1] = np.clip(ref_pred, 0, None)
ref_fname = "submission_LGB35x075_PLS2s0x025_BEST.csv"
ref_out.to_csv(ref_fname, index=False, header=False)
all_subs[ref_fname] = 16.67
print(f"  ✅ {ref_fname} (LB=11.428 再現)")

# Top5 2モデルブレンド（重複排除）
seen = set()
count = 0
for b in blend2_results:
    key = (b['lgb_seed'], b['pls_nc'], b['pls_seed'])
    if key in seen:
        continue
    seen.add(key)
    out = submit_template.copy()
    out[1] = np.clip(b['test_pred'], 0, None)
    safe = (b['name'].replace("×", "x").replace("+", "_")
            .replace(".", "p"))
    fname = f"submission_2bl_{safe}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = b['oof']
    print(f"  ✅ {fname} (OOF={b['oof']:.4f})")
    count += 1
    if count >= 8:
        break

# Top5 3モデルブレンド
seen3 = set()
count3 = 0
for b in blend3_results:
    key = (b['seeds'], b['pls'])
    if key in seen3:
        continue
    seen3.add(key)
    out = submit_template.copy()
    out[1] = np.clip(b['test_pred'], 0, None)
    safe = (b['name'].replace("×", "x").replace("+", "_")
            .replace(".", "p"))
    fname = f"submission_3bl_{safe}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = b['oof']
    print(f"  ✅ {fname} (OOF={b['oof']:.4f})")
    count3 += 1
    if count3 >= 5:
        break

# w精密探索のベスト（LGB35+PLS2_s0の最適w）
lgb35_pls0_blends = [b for b in blend2_results
                     if b['lgb_seed'] == 35 and
                     b['pls_nc'] == 2 and b['pls_seed'] == 0]
if lgb35_pls0_blends:
    best_w = lgb35_pls0_blends[0]
    out = submit_template.copy()
    out[1] = np.clip(best_w['test_pred'], 0, None)
    fname = f"submission_LGB35_PLS2s0_optw{best_w['w']:.2f}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = best_w['oof']
    print(f"  ✅ {fname} (OOF={best_w['oof']:.4f}, w={best_w['w']:.2f})")
    print(f"     → LGB35+PLS2_s0の最適w = {best_w['w']:.2f} "
          f"(11.428は w=0.75)")


# ============================================================
# 9. 全スコア比較
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較")
print(f"{'='*60}")
print(f"  {'手法':<45s} {'OOF':>8s} {'LB':>8s}")
print(f"  {'─'*45} {'─'*8} {'─'*8}")

for name, oof, lb in HISTORY:
    oof_s = f"{oof:.2f}" if oof is not None else "---"
    m = " ★BEST" if lb == 11.428 else ""
    print(f"  {name:<45s} {oof_s:>8s} {lb:.3f}{m}")

print(f"  {'─'*45} {'─'*8} {'─'*8}")

for fname, oof in sorted(all_subs.items(), key=lambda x: x[1])[:15]:
    short = fname.replace("submission_", "").replace(".csv", "")
    print(f"  {short:<45s} {oof:>8.2f} {'???':>8s}")


# ============================================================
# 10. 提出判断
# ============================================================
print(f"\n{'='*60}")
print("🧭 提出判断ガイド")
print(f"{'='*60}")

# LGB35+PLS2_s0の最適w
opt = lgb35_pls0_blends[0] if lgb35_pls0_blends else None

print(f"""
  ■ 確定事実:
    LGB35×0.75 + PLS2_s0×0.25 = LB 11.428 ★BEST
    
  ■ 今回判明した最適w:
    LGB35 + PLS2_s0: w = {opt['w']:.2f} (OOF={opt['oof']:.4f})
    vs w=0.75: OOF差 = {opt['oof'] - 16.67:.4f}
    
  ■ 2モデル vs 3モデル:
    ベスト2モデル: OOF = {best2['oof']:.4f}
    ベスト3モデル: OOF = {best3['oof']:.4f}
""")

# 推奨
print(f"  ■ 推奨提出順:")

priority = []

# 1. 最適w
if opt and abs(opt['w'] - 0.75) > 0.005:
    priority.append((
        f"submission_LGB35_PLS2s0_optw{opt['w']:.2f}.csv",
        f"w最適化 (w={opt['w']:.2f} vs 0.75), OOF={opt['oof']:.4f}"
    ))

# 2. 異なるLGBseedのベストブレンド
non35_blends = [b for b in blend2_results
                if b['lgb_seed'] != 35]
if non35_blends:
    best_non35 = non35_blends[0]
    safe = (best_non35['name'].replace("×", "x").replace("+", "_")
            .replace(".", "p"))
    priority.append((
        f"submission_2bl_{safe}.csv",
        f"異なるLGB seed, OOF={best_non35['oof']:.4f}"
    ))

# 3. ベスト3モデルブレンド
safe3 = (best3['name'].replace("×", "x").replace("+", "_")
         .replace(".", "p"))
priority.append((
    f"submission_3bl_{safe3}.csv",
    f"3モデルブレンド, OOF={best3['oof']:.4f}"
))

# 4. 異なるPLS成分のブレンド
pls1_blends = [b for b in blend2_results if b['pls_nc'] == 1]
if pls1_blends:
    best_pls1 = pls1_blends[0]
    priority.append((
        f"PLS1ブレンド",
        f"PLS1成分, OOF={best_pls1['oof']:.4f}"
    ))

for i, (fname, reason) in enumerate(priority[:4]):
    print(f"    {i+1}. {fname}")
    print(f"       理由: {reason}")

print(f"""
  ■ 次回以降の方向:
    - w最適化で改善 → さらにLGB seedの広域探索
    - 3モデルで改善 → 4モデル (LGB×2 + PLS×2)
    - PLS1成分で改善 → 超シンプルモデルの可能性
    - 上記すべて不変 → 1D-CNN等の構造変更
""")

🏆 ブレンド精密最適化

  ★ LB=11.428 (LGB35×0.75 + PLS2_s0×0.25)

  今回の目標:
  1. w=0.70~0.80を0.01刻みで精密探索
  2. 異なるPLS seed/成分数でのブレンド
  3. 3モデルブレンド (LGB35 + LGB_X + PLS)
  4. 異なるLGB seedでのブレンド


🔧 ベースモデル構築
  LGB seed=  35  OOF=16.7930
  LGB seed=  39  OOF=17.5141
  LGB seed=  33  OOF=17.5727
  LGB seed=  42  OOF=18.0379
  LGB seed=  25  OOF=17.6162
  LGB seed=  29  OOF=17.6201
  LGB seed=  34  OOF=17.6726
  LGB seed=  28  OOF=17.6937
  PLS2 seed=   0  OOF=19.5861
  PLS2 seed=  42  OOF=19.6025
  PLS2 seed=  33  OOF=19.9639
  PLS2 seed=  35  OOF=20.6136
  PLS2 seed=  38  OOF=20.0582
  PLS1 seed=   0  OOF=16.8126
  PLS1 seed=  42  OOF=16.7626
  PLS3 seed=   0  OOF=20.1120
  PLS3 seed=  42  OOF=20.3538

🔬 実験1: 2モデルブレンド精密探索
  全2304組み合わせを探索

  Top 20:
  Name                                               OOF
  ───────────────────────────────────────────── ────────
  LGB35×0.60+PLS1_s42×0.40                       15.5732
  LGB35×0.61+PLS1_s42×0.39                       15.5845
  LGB35×0.62+PLS1_s42×0.38   